# 36 · The enterprise map: tenancy, managed runtimes, governance and HA/DR
Level L6 · Part 6 · professional · ~20 min · builds on 30, 35

A walkthrough. The code is standard library only, with no model, no key and no cloud account. The managed runtimes
are described from their official docs, because you need an account to click through them.

## Goal
Place the Brightside support desk in an organisation. For each question (whose data is this, where does the agent run,
which law applies, what do we lose if the server dies), name the boundary and the number that answers it.

## Why it matters
Lessons 30 to 35 made one agent safe, cheap, durable and fast. A company runs many agents for many customers, though,
and then the questions change. Without a map, four things happen:
- One customer's data reaches another. A cache keyed by the question alone serves the same answer to everyone
  (the caches of lesson 33 need the same care about keys).
- A team picks a runtime by habit, then finds the session limit at 2 a.m.
- The first EU customer asks which AI Act tier the desk is in, and nobody can say.
- A worker dies in the middle of a refund run, and the run starts again from nothing.

The cell below shows the first problem with the code you already know. Then the fix.

## The idea
An enterprise agent has four boundaries. **Tenancy** keeps each customer's data, memory, cache and tool credentials
apart. A **managed runtime** runs the agent in a sandbox per session, so you do not run the servers yourself.
**Governance** names the rules the agent must meet and who owns each one. **HA/DR** says how much work you can lose
(RPO) and how long the desk can be down (RTO). Each boundary has a test you can write down, and a number you can check.

Analogy: a shopping centre. Each shop has its own door and its own safe (tenancy). The centre rents the floor space and
security (the managed runtime). The building code says what each shop may do (governance). The fire plan says how fast
the shops reopen and how many sales are lost (HA/DR).

## Picture
```mermaid
flowchart LR
  B[Brightside shopper] --> G[Gateway: who is calling, which tenant, budget]
  N[Northfield shopper] --> G
  G --> S1[Runtime session: Brightside conversation]
  G --> S2[Runtime session: Northfield conversation]
  S1 --> M1[(Memory and cache keyed by tenant)]
  S2 --> M2[(Memory and cache keyed by tenant)]
  S1 --> T[Tools with per-tenant credentials]
  S2 --> T
  S1 -. checkpoint .-> D[(Durable store, lesson 35)]
  S2 -. checkpoint .-> D
```

## Step by step
**Step 1 · Tenancy: run the leak.** The desk serves two retailers. Brightside and Northfield are fictional, and each
has its own orders. First, the shared answer cache with a key that has no tenant in it. You should see Northfield
receive Brightside's order status.

In [1]:
# Fictional orders for two tenants. Standard library only: no model, no key.
ORDERS = {  # order id -> (tenant, status)
    "1042": ("brightside", "left the depot on Monday"),
    "77": ("northfield", "with the courier in Leeds"),
}

answer_cache: dict[str, str] = {}  # the desk's shared cache: one entry per question


def ask_bad(tenant: str, question: str, order_id: str) -> str:  # tenant is passed in but never used: that is the bug
    if question in answer_cache:  # the bug: the key has no tenant in it
        return answer_cache[question]
    status = ORDERS[order_id][1]  # the lookup does not check the tenant either
    answer_cache[question] = f"Order {order_id} is {status}."
    return answer_cache[question]


print("brightside:", ask_bad("brightside", "where is my parcel?", "1042"))
print("northfield:", ask_bad("northfield", "where is my parcel?", "77"))  # the same question, another tenant

brightside: Order 1042 is left the depot on Monday.
northfield: Order 1042 is left the depot on Monday.


**Step 1 (fixed).** Two changes. The cache key holds the tenant, and the lookup checks that the order belongs to the
caller. The tenant is then a boundary the code enforces, not a label. You should see each tenant get its own answer.

In [2]:
def lookup(tenant: str, order_id: str) -> str:
    owner, status = ORDERS[order_id]
    if owner != tenant:  # the check that makes the tenant a boundary
        raise PermissionError(f"order {order_id} is not in tenant {tenant}")
    return f"Order {order_id} is {status}."


good_cache: dict[tuple[str, str], str] = {}  # the key is (tenant, question)


def ask_good(tenant: str, question: str, order_id: str) -> str:
    key = (tenant, question)
    if key not in good_cache:
        good_cache[key] = lookup(tenant, order_id)
    return good_cache[key]


print("brightside:", ask_good("brightside", "where is my parcel?", "1042"))
print("northfield:", ask_good("northfield", "where is my parcel?", "77"))

brightside: Order 1042 is left the depot on Monday.
northfield: Order 77 is with the courier in Leeds.


**Step 2 · The session boundary (API walkthrough, not run here).** A managed runtime gives each session its own
sandbox. For Amazon Bedrock AgentCore Runtime the official docs say: each session gets its own dedicated microVM (a tiny virtual
machine, isolated from the others), with
its own compute, memory and filesystem; when the session ends the microVM is terminated and its memory is sanitised.
The session stops after 15 minutes idle by default, and after at most 8 hours of compute. The API equivalent is one
call per turn, with the same session id for one conversation:

```text
InvokeAgentRuntime(agentRuntimeArn=..., runtimeSessionId=conversation_id, payload=...)
```

**Read the fine print.** AgentCore does not map users to sessions. Your backend must, and must refuse a session id
that belongs to another user. The session id must be at least 33 characters long.

**Step 3 · Choose where the agent runs.** Names changed in 2026: check
each docs page. *Snapshot, verified 2026-10-21.*

| Option | What the platform gives you | What you still own | Verified / caveat |
|---|---|---|---|
| Your own container (Cloud Run, ECS, Container Apps) | scaling, HTTPS, the cloud account | sandbox, sessions, on-call | lessons 32-35 |
| Amazon Bedrock AgentCore Runtime | a microVM per session, identity, gateway | the session-to-user map, tools | limits above |
| Gemini Enterprise Agent Platform, Agent Runtime | renamed from Vertex AI Agent Engine (2026-04-22) | agent code, tenant rules | isolation: not verified here |
| Microsoft Foundry Agent Service, hosted agents | your container, a Microsoft Entra identity per agent, a VM-isolated sandbox per session | agent logic, tenant rules | GA reported in an Azure Update dated 2026-06-02; the docs overview label was not confirmed here, so read it before you rely on GA |

**What no managed runtime removes:** the tenant check in your tools, the approval for a refund, and the trace that
shows which tenant a call belonged to. Those stay in your code (lessons 31 and 35).

**Step 4 · Governance: the vocabulary.** You do not need to be a lawyer to place the desk. You need each name, what it
is and what it is not. Whether the desk is high-risk under the EU AI Act is a legal question: ask counsel.
*Snapshot, verified on 2026-10-21. Annex III is the AI Act's list of high-risk areas, such as hiring or credit scoring.*

| Name | What it is | What it is not |
|---|---|---|
| EU AI Act (Regulation (EU) 2024/1689) | binding law with risk tiers | a checklist you pass once |
| Digital Omnibus on AI, Regulation (EU) 2026/1744 (dates below come from secondary sources (Hunton, fasi.eu); confirm in EUR-Lex before relying on them) | moves dates: Annex III high-risk duties to 2 Dec 2027, Annex I to 2 Aug 2028; published in the Official Journal 24 Jul 2026 | a repeal of the AI Act |
| Art. 50 transparency (AI Act) | applies from 2 Aug 2026; marking for systems already on the market has a grace period to 2 Dec 2026 (grace date from a pre-adoption source; check EUR-Lex) | a rule for every chatbot answer |
Dates in this table were not read from EUR-Lex.

| Name | What it is | What it is not |
|---|---|---|
| ISO/IEC 42001:2023 | a management-system standard for AI; an organisation can be certified | a certificate for one agent or one model |
| NIST AI RMF 1.0 | a voluntary framework: Govern, Map, Measure, Manage; the GenAI profile (NIST AI 600-1, 26 Jul 2024) adds generative-AI risks | a law |
| Data residency | where personal data is stored and processed | the same as availability: a region can be up and still break a residency rule |

A desk that answers "where is my parcel?" is often treated as limited-risk. It becomes high-risk when the agent decides
eligibility in an Annex III area (the AI Act's list of high-risk areas, such as hiring or credit). That judgement is for counsel, and the risk register is where you write it down.

**Step 5 · HA/DR: the two numbers.** **RPO** is the most recent data you can afford to lose, measured as the age of the
last good copy at the moment of failure. **RTO** is the longest the service may be down. The business sets both, and
the technical team picks the design that meets them (AWS Well-Architected, reliability pillar). The refund run from
lesson 35 saves a record after each step, so the numbers below are the worst case for it. They are assumptions to
replace with your own measurements.

In [3]:
checkpoint_every_s = 3   # our choice, not from lesson 35: a step's result is saved at least this often (assumption)
new_worker_s = 30        # time to start a worker elsewhere and read the record (assumption)
timeout_s = 3            # the start-to-close timeout from lesson 35 (the longest one step may run): how long until the crash is noticed
rpo_s = checkpoint_every_s         # the most work lost: everything since the last save; a step is one tool or model call, and the 3 s assumes a step ends within about 3 s (a longer step can lose more)
rto_s = timeout_s + new_worker_s   # RTO = detection time (the timeout) + restart time (starting a worker and reading the record)
print(f"RPO (work lost at worst): {rpo_s} s · RTO (crash to resumed run): {rto_s} s")

RPO (work lost at worst): 3 s · RTO (crash to resumed run): 33 s


A step that moves money is the exception. If the worker dies after the charge but before the save, the charge runs
again on resume. The idempotency key from lesson 32 stops the second charge, so the run resumes safely. "Zero data
loss" for every record is a target AWS warns is often unachievable; set the RPO per record type.

## At real scale
- **Access is decided per tool, not per server.** Uber's MCP gateway (2026-10-01) runs 800+ MCP servers and 5,000+ tools. Tools start
  disabled and get a per-tool access policy (https://www.uber.com/us/en/blog/designing-mcp-gateway/). → lesson 26
- **Each hop carries a short-lived identity.** Uber's agent identity (2026-05-21) uses per-hop tokens with an actor
  chain; its token service answers in p99 under 40 ms (https://www.uber.com/us/en/blog/solving-the-agent-identity-crisis/). → lesson 31
- **A session is a unit of cost and of loss.** AgentCore's 15-minute idle default ends a session. A refund that waits
  longer must live in a durable run. → lesson 35
- **Residency comes before the endpoint.** A regional endpoint may cost more. For regulated data, residency wins over
  availability from another region. → lesson 33 for cost

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| Several customers use one agent | a tenant id in every span, cache key, memory row and tool credential | one missing key is a leak; the step 1 cell is the test |
| Each conversation needs its own sandbox | a managed per-session runtime, or your own sandbox per session | the platform gives isolation; you still map users to sessions |
| Your team already runs containers and a queue | your own container, with the patterns from lessons 32 to 35 | you keep control, and the on-call |

| Situation | Approach | Why |
|---|---|---|
| Regulated data must stay in one region | a regional endpoint and a runtime in that region, checked first | residency beats availability for regulated data |
| A refund waits days for approval | a durable run with a checkpoint per step | a session timeout ends the wait; the record does not |
| A customer asks for the AI Act tier | a risk register with the Annex III question and counsel's answer | the tier depends on what the agent decides, not on the model |

## Common mistakes
**Asking for another tenant's order.** Step 1 showed a cache leak. This is the same mistake through the lookup: without
the owner check, Northfield can read Brightside's order by its id. With the check, the call fails, as it should.

In [4]:
print(lookup("northfield", "1042"))  # Brightside's order, asked for by Northfield: the check stops it

PermissionError: order 1042 is not in tenant northfield

The fix is the owner check in `lookup` and the tenant in the cache key. Two more mistakes: calling an ISO/IEC 42001
certificate "AI Act compliance" (it covers the organisation's management system, not the law), and setting one RPO and
RTO for the whole desk instead of one per record type.

## Try it
Add a monthly budget per tenant. A call is refused before it runs when the tenant's spend would pass its cap. Keep the
spend per tenant in a dict. Test it with Brightside at a cap of 5.00 and two calls of 3.00 each. The first call should
pass and the second should be refused.

**Answer** — try it first, then expand the hidden cell below.

In [5]:
spend: dict[str, float] = {}  # tenant -> USD spent so far
CAP = {"brightside": 5.00}


def charge_budget(tenant: str, cost: float) -> bool:
    new_total = spend.get(tenant, 0.0) + cost
    if new_total > CAP[tenant]:  # refuse before the call runs, not after
        return False
    spend[tenant] = new_total
    return True


print(charge_budget("brightside", 3.00))  # 3.00 of 5.00: allowed
print(charge_budget("brightside", 3.00))  # would be 6.00: refused

True
False


## Recap
- A tenant is a boundary the code enforces: the tenant goes in every cache key, memory row and tool credential, and every
  lookup checks the owner. The step 1 cells show both the leak and the fix.
- A managed runtime gives each session its own sandbox, with time limits (AgentCore: 15 minutes idle, 8 hours of
  compute by default). Your backend still maps users to sessions.
- Governance is vocabulary you can place: the AI Act and its 2026 date changes (confirm in EUR-Lex), ISO/IEC 42001 for the management
  system, NIST AI RMF for risk, and residency for where data sits. Tier questions go to counsel.
- RPO is the work you can lose, RTO is the downtime you can take. Set both per record type; a money step is safe
  only with an idempotency key.

**Go deeper:** agent identity (lesson 31), the MCP gateway (lesson 26), cost (lesson 33), durable runs (lesson 35).

**Interview:** "A company wants one agent for a hundred customers. What do you isolate, and how do you prove it?"
Strong answer: "A tenant id in every span, cache key, memory row and tool credential, and an owner check on every
lookup. A test asks for another tenant's data and expects a refusal. Per-session sandboxes for code and files, with
the user-to-session map in my backend."